# Лабораторная 04. Spark UI как инструмент диагностики

Цель: научиться читать Jobs, Stages, SQL и Executors tabs.

In [7]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-04-spark-ui').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base = Path('spark_core_data').absolute()
base_uri = base.as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Spark UI: http://0a370e2ebe67:4040


## Операция 1: count
Запустите action и заполните Jobs/Stages.

In [8]:
orders.count()

120000

## Операция 2: groupBy
Смотрите Shuffle Read/Write и SQL plan.

In [9]:
orders.groupBy('status').agg(F.count('*').alias('cnt'), F.sum('order_amount').alias('amount')).show()

+---------+-----+----------+
|   status|  cnt|    amount|
+---------+-----+----------+
|     paid|30000|8117202.62|
|  shipped|30000|8086082.66|
|cancelled|30000|8128236.05|
|  created|30000|8105749.92|
+---------+-----+----------+



## Операция 3: join
Отключим broadcast, чтобы план был показательнее.

In [10]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
orders.join(customers, 'customer_id').groupBy('region').count().show()

+-------+-----+
| region|count|
+-------+-----+
|   east|24000|
|  north|24000|
|   west|24000|
|central|24000|
|  south|24000|
+-------+-----+



## Операция 4: repartition + write
Запись тоже action. Количество output files связано с количеством partitions.

In [11]:
out = base / 'ui_lab_output'
if out.exists():
    shutil.rmtree(out)
orders.repartition(6).write.mode('overwrite').parquet(out.as_uri())
out.as_uri()

'file:///materials/seminar_04_spark_core/practice/spark_core_data/ui_lab_output'

Заполните для каждой операции:

| Операция | Job ID | Stages | Duration | Самый долгий stage | Tasks в нем | Shuffle Read/Write в нем | Spill | Оператор в SQL plan |
|---|---|---:|---:|---|---:|---|---|---|
| count | 2 | 2,3 | 0.3s | #2 (0.1 s) | Scan parquet, WholeStageCodegen, Exchange | 236.0 B	 | ? | HashAggregate |
| groupBy | 3 | 4,5 | 13s | #4 (0.8 s) | Scan parquet, WholeStageCodegen, Exchange | 1336.0 B| ? | HashAggregate |
| join | 6 | 10,11,12,13 | 2s | #12 (1.0s) | exchange x2 > wholestagecodegen x2 >  wholestagecodegen > exchange | read 668.4 KiB; write  2.4 KiB| ? | SortMergeJoin Inner |
| repartition/write | 9 | 22,23 | 8s | #23 (7s) | exchange, writefiles | 2.5 MiB | ? | Exchange/ WriteFiles |

Итоговый мини-отчёт 5-7 предложений:

```text
Самым дорогим оказался write с repartition (Job 9) - 8 секунд. 
Он дорогой, потому что требует полной перетасовки данных (repartition на 6 партиций)
и физической записи на диск в формате Parquet.

На втором месте join (Job 6) - 2 секунды. 
Он требует двух перетасовок для подготовки данных, сортировки и самого SortMergeJoin.

GroupBy (Job 3) выполняется 1 секунду, а count (Job 2) - всего 0.3 секунды,
так как count без группировки требует только одной перетасовки и агрегации.

```

In [12]:
spark.stop()